In [5]:
# ── Dependency Guard ───────────────────────────────────────────────
import importlib, subprocess, sys

_REQUIRED = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'sklearn': 'scikit-learn',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'fastf1': 'fastf1',
}

_missing = []
for _mod, _pip in _REQUIRED.items():
    try:
        importlib.import_module(_mod)
    except ModuleNotFoundError:
        _missing.append(_pip)

if _missing:
    print(f'Installing missing packages: {_missing}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet'] + _missing)
    print('Done. Packages installed successfully.')
else:
    print('All required packages already installed ✓')

# ── Library Imports ───────────────────────────────────────────────
import os                        # Working directory checks
import subprocess                # Git command checks
import importlib                 # Runtime dependency checks
import numpy as np               # Numeric support
import pandas as pd              # Tables and diagnostics
import matplotlib.pyplot as plt  # Plotting
import seaborn as sns            # Statistical plotting
import fastf1                    # Formula 1 data access

print(f'fastf1  : {fastf1.__version__}')
print(f'pandas  : {pd.__version__}')

All required packages already installed ✓
fastf1  : 3.8.1
pandas  : 2.3.3


In [6]:
# ── Reproducibility Header ────────────────────────────────────────────

import sys, random
import numpy as np
import warnings

RANDOM_SEED = 414  # Course constant. Do not change.
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
warnings.filterwarnings('ignore', category=FutureWarning)

# Environment check
print(f'Python  : {sys.version.split()[0]}')
print(f'NumPy   : {np.__version__}')
print(f'Seed    : {RANDOM_SEED}')

Python  : 3.12.3
NumPy   : 2.4.3
Seed    : 414


In [ ]:
# ── Rule-Based Prediction: Grid Position → DNF Prediction ────────────────

# Get race data from 2023 season (validation set)
# Using races from mid-season onwards
season = 2023
races_to_fetch = [10, 11, 12, 13, 14, 15]  # Races 10-15

predictions = []
actual_results = []
race_info = []

print(f"Fetching {len(races_to_fetch)} races from {season} season...")
print("=" * 70)

for race_round in races_to_fetch:
    try:
        session = fastf1.get_session(season, race_round, 'R')  # 'R' for race
        session.load()
        
        race_name = session.event['EventName']
        
        # Get results dataframe with grid and DNF status
        results = session.results
        
        # Identify DNF drivers (Status is not 'Finished')
        dnf_drivers = set(results[results['Status'] != 'Finished'].index)
        finished_drivers = set(results[results['Status'] == 'Finished'].index)
        
        # Apply rule: grid > 14 → predict DNF, grid <= 14 → predict finish
        for driver_idx in results.index:
            grid_pos = results.loc[driver_idx, 'GridPosition']
            
            # Skip if no valid grid position (sometimes NaN for reserve/safety car)
            if pd.isna(grid_pos):
                continue
            
            grid_pos = int(grid_pos)
            
            # Apply rule
            predicted_dnf = grid_pos > 14
            actual_dnf = driver_idx in dnf_drivers
            
            is_correct = predicted_dnf == actual_dnf
            
            predictions.append({
                'race': race_name,
                'round': race_round,
                'driver': driver_idx,
                'grid_position': grid_pos,
                'predicted_dnf': predicted_dnf,
                'actual_dnf': actual_dnf,
                'correct': is_correct
            })
        
        print(f"✓ Round {race_round}: {race_name} - {len(results)} drivers")
        
    except Exception as e:
        print(f"✗ Round {race_round}: Error - {str(e)}")

# Convert to DataFrame for analysis
predictions_df = pd.DataFrame(predictions)

# Calculate metrics
total_predictions = len(predictions_df)
correct_predictions = predictions_df['correct'].sum()
accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0

print("=" * 70)
print(f"\n📊 RULE-BASED PREDICTION RESULTS")
print(f"{'─' * 70}")
print(f"Rule: If GridPosition > 14 → Predict DNF | Otherwise → Predict Finish")
print(f"{'─' * 70}")
print(f"Total Predictions:  {total_predictions}")
print(f"Correct:            {correct_predictions}")
print(f"Incorrect:          {total_predictions - correct_predictions}")
print(f"\n🎯 ACCURACY: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"{'─' * 70}")

# Breakdown by outcome
print(f"\nDetailed Breakdown:")
print(f"  - Correctly predicted DNF:    {predictions_df[(predictions_df['predicted_dnf'] == True) & (predictions_df['correct'] == True)].shape[0]}")
print(f"  - Correctly predicted Finish: {predictions_df[(predictions_df['predicted_dnf'] == False) & (predictions_df['correct'] == True)].shape[0]}")
print(f"  - False positives (DNF):      {predictions_df[(predictions_df['predicted_dnf'] == True) & (predictions_df['correct'] == False)].shape[0]}")
print(f"  - False negatives (Finish):   {predictions_df[(predictions_df['predicted_dnf'] == False) & (predictions_df['correct'] == False)].shape[0]}")

The acuarcy of the method was 67.23% for the lab 2 we need to at least get that accurate.

## 4.3 Reflection on Baseline Accuracy

### Is this accuracy good enough to make decisions with?

The baseline accuracy of **{accuracy*100:.2f}%** tells us how well a simple grid position heuristic performs. However, accuracy alone can be misleading:

- **Pros**: This simple rule requires no machine learning and is immediately interpretable — anyone can understand "grid position > 14 predicts DNF."
- **Cons**: Accuracy doesn't tell us about failure modes. For example:
  - What if 50% of drivers finish in the top 10? Then a "always predict finish" baseline would score 50% accuracy without any predictive power.
  - We're ignoring driver skill, car reliability, weather, track characteristics, and pit stop strategy.

### What could accuracy be hiding?

1. **Class Imbalance**: If only 10% of drivers DNF, a naive model predicting "always finish" gets 90% accuracy while being useless.
   - *Our rule addresses this partially by using grid position as a proxy.*

2. **Actionability**: A 60% accurate predictor might be useless for betting or strategy if it's only slightly better than random guessing (50%).
   - *We should compare against random guessing and the baseline of "always predict the class majority."*

3. **False Positive vs False Negative Cost**: 
   - **False Positive (predicting DNF but driver finishes)**: Strategy team wastes resources planning for a DNF.
   - **False Negative (predicting finish but driver DNFs)**: Strategy team is caught off-guard.
   - These may have different business costs, but raw accuracy treats them equally.

### Key Insight for Lab 2

To beat the Lab 2 accuracy of **67.23%**, we need to incorporate additional features beyond grid position—such as:
- Driver historical performance
- Team reliability metrics
- Weather conditions
- Track suitability
- Qualifying gap analysis

The current baseline is a **sanity check**, not a production model.